In [2]:
import pandas as pd
survey = pd.read_csv("../data/raw/statcan_13100972.csv", encoding="utf-8-sig")
print(survey.shape)
print(survey.columns.tolist())
survey["Characteristics"].unique()[:40]

(2042150, 18)
['REF_DATE', 'GEO', 'DGUID', 'Age group', 'Sex', 'Indicators', 'Characteristics', 'UOM', 'UOM_ID', 'SCALAR_FACTOR', 'SCALAR_ID', 'VECTOR', 'COORDINATE', 'VALUE', 'STATUS', 'SYMBOL', 'TERMINATED', 'DECIMALS']


<StringArray>
[                                     'Number of persons',
         'Low 95% confidence interval, number of persons',
        'High 95% confidence interval, number of persons',
                                                'Percent',
                   'Low 95% confidence interval, percent',
                  'High 95% confidence interval, percent',
 'Statistically different from previous reference period',
           'Statistically different from the Canada rate',
       'Statistically different from the provincial rate']
Length: 9, dtype: str

In [3]:
# inspecting
print(survey["REF_DATE"].unique())
print(survey["Age group"].unique())
print(survey["Sex"].unique())
print(survey["Indicators"].unique())

s = survey[(survey["REF_DATE"] == "2023/2024") & (survey["Characteristics"] == "Percent")] # only keeps percent values
print(s["GEO"].nunique())
print(s[s["GEO"].str.contains("Alberta|Edmonton|Calgary|Corridor|Zone", case=False)]["GEO"].unique()) # checks region 

<StringArray>
['2015/2016', '2017/2018', '2019/2020', '2021/2022', '2023/2024']
Length: 5, dtype: str
<StringArray>
['Total, 18 years and over',           '18 to 34 years',
           '35 to 49 years',           '50 to 64 years',
        '65 years and over']
Length: 5, dtype: str
<StringArray>
['Both sexes', 'Males', 'Females']
Length: 3, dtype: str
<StringArray>
[                              'Perceived health, very good or excellent',
                                         'Perceived health, fair or poor',
                        'Perceived mental health, very good or excellent',
                                  'Perceived mental health, fair or poor',
    'Perceived life stress, most days quite a bit or extremely stressful',
                    'Body mass index, adjusted self-reported, overweight',
                         'Body mass index, adjusted self-reported, obese',
                                                              'Arthritis',
                                  

We can use certain indicators from this table as our predictors which could tell us more about the health need a region has and therefore if it affects the wait times.
Arthritis- leading reason for hip replacement
Perceived health, fair or poor
BMI, obese- factor for joint replacement

In [4]:
keep = ["Arthritis", "Perceived health, fair or poor", "Body mass index, adjusted self-reported, obese"]

s = survey[
    (survey["REF_DATE"] == "2023/2024")
    & (survey["Characteristics"] == "Percent")
    & (survey["Age group"] == "Total, 18 years and over")
    & (survey["Sex"] == "Both sexes")
    & (survey["Indicators"].isin(keep))
]

wide = s.pivot_table(index="GEO", columns="Indicators", values="VALUE").reset_index()
wide.columns = ["geo", "arthritis_pct", "fair_poor_health_pct", "obese_pct"]
wide[["region", "province"]] = wide["geo"].str.rsplit(", ", n=1, expand=True)

print(wide.shape)
print(wide.isna().sum())
wide[wide["geo"].str.contains("Edmonton")]

(112, 6)
geo                      0
arthritis_pct            0
fair_poor_health_pct     0
obese_pct                0
region                   0
province                14
dtype: int64


,geo,arthritis_pct,fair_poor_health_pct,obese_pct,region,province
18,"Edmonton Zone, Alberta",21.4,33.1,15.0,Edmonton Zone,Alberta


In [5]:
regions = pd.read_csv("../data/processed/regions_2024.csv")
regions[["province", "region"]].head(60)

,province,region
0,Alberta,Calgary Zone
1,Alberta,Central Zone (Alta.)
2,Alberta,Edmonton Zone
3,Alberta,North Zone (Alta.)
4,Alberta,South Zone
5,British Columbia,Fraser Health
6,British Columbia,Interior Health
7,British Columbia,Island Health
8,British Columbia,Northern Health
9,British Columbia,Vancouver Coastal Health


In [6]:
import re, unicodedata
# cleaning and checking for name differences 
def make_key(s):
    s = s.lower()
    s = re.sub(r"\(.*?\)", "", s)
    s = re.sub(r"zone \d+\s*[-–]\s*", "", s)
    s = re.sub(r"[-–—]", " ", s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    s = re.sub(r"\b(zone|region|health|authority)\b", "", s)
    return re.sub(r"\s+", " ", s).strip()

wide = wide.dropna(subset=["province"]).rename(columns={"region": "survey_region"})
wide["key"] = wide["survey_region"].map(make_key)
regions["key"] = regions["region"].map(make_key)

m = regions.merge(
    wide[["province", "key", "survey_region", "arthritis_pct", "fair_poor_health_pct", "obese_pct"]],
    on=["province", "key"], how="outer", indicator=True
)
print(m["_merge"].value_counts())
print("Ours, no survey match:")
print(m[m["_merge"] == "left_only"][["province", "region"]])
print("Survey, no match (our provinces only):")
print(m[(m["_merge"] == "right_only") & m["province"].isin(regions["province"])][["province", "survey_region"]])

_merge
right_only    76
left_only     27
both          22
Name: count, dtype: int64
Ours, no survey match:
                 province                                region
7        British Columbia                         Fraser Health
11       British Columbia                       Interior Health
12       British Columbia                         Island Health
17       British Columbia                       Northern Health
24       British Columbia              Vancouver Coastal Health
48                Ontario                        Central Region
55                Ontario                           East Region
69                Ontario                     North East Region
70                Ontario                     North West Region
81                Ontario                        Toronto Region
84                Ontario                           West Region
87   Prince Edward Island                            Health PEI
88                 Quebec          Abitibi-Témiscamingue Regi

In [7]:
# to check spellings for provinces that didnt match
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)
for p in ["Ontario", "Prince Edward Island", "British Columbia", "Quebec"]:
    print(p)
    print(wide[wide["province"] == p]["survey_region"].tolist())
    print()

Ontario
['Brant County Health Unit', 'Chatham-Kent Health Unit', 'City of Hamilton Health Unit', 'City of Ottawa Health Unit', 'City of Toronto Health Unit', 'District of Algoma Health Unit', 'Durham Regional Health Unit', 'Eastern Ontario Health Unit', 'Grey Bruce Health Unit', 'Haldimand-Norfolk Health Unit', 'Haliburton, Kawartha, Pine Ridge District Health Unit', 'Halton Regional Health Unit', 'Hastings and Prince Edward Counties Health Unit', 'Huron Perth Health Unit', 'Kingston, Frontenac and Lennox and Addington Health Unit', 'Lambton Health Unit', 'Leeds, Grenville and Lanark District Health Unit', 'Middlesex-London Health Unit', 'Niagara Regional Area Health Unit', 'North Bay Parry Sound District Health Unit', 'Northwestern Health Unit', 'Oxford Elgin St. Thomas Health Unit', 'Peel Regional Health Unit', 'Peterborough County-City Health Unit', 'Porcupine Health Unit', 'Renfrew County and District Health Unit', 'Simcoe Muskoka District Health Unit', 'Sudbury and District Health

In [11]:
def make_key(s): # fixing quebec french spellings
    s = s.lower()
    s = re.sub(r"\(.*?\)", "", s)
    s = re.sub(r"zone \d+\s*[-–]\s*", "", s)
    s = re.sub(r"[-–—]", " ", s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    s = re.sub(r"\bregion (des|du|de la|de l'|de)\s+", "", s)
    s = re.sub(r"\bl'", "", s)
    s = re.sub(r"\bet du\b", "et", s)
    s = re.sub(r"\b(zone|region|health|authority)\b", "", s)
    return re.sub(r"\s+", " ", s).strip()

base = survey[
    (survey["REF_DATE"] == "2023/2024")
    & (survey["Age group"] == "Total, 18 years and over")
    & (survey["Sex"] == "Both sexes")
    & (survey["Indicators"].isin(keep))
]
p = base[base["Characteristics"] == "Percent"].pivot_table(index="GEO", columns="Indicators", values="VALUE")
n = base[base["Characteristics"] == "Number of persons"].pivot_table(index="GEO", columns="Indicators", values="VALUE")

def combine(groups, province, suffix=""):
    rows = []
    for target, names in groups.items():
        geos = [f"{x}{suffix}, {province}" for x in names]
        count = n.loc[geos].sum()
        pop = (n.loc[geos] / (p.loc[geos] / 100)).sum()
        rows.append({"province": province, "region": target, **(100 * count / pop).to_dict()})
    return rows

bc = {
    "Interior Health": ["East Kootenay", "Kootenay-Boundary", "Okanagan", "Thompson/Cariboo"],
    "Fraser Health": ["Fraser East", "Fraser North", "Fraser South"],
    "Vancouver Coastal Health": ["Richmond", "Vancouver", "North Shore/Coast Garibaldi"],
    "Island Health": ["South Vancouver Island", "Central Vancouver Island", "North Vancouver Island"],
    "Northern Health": ["Northwest", "Northern Interior", "Northeast"],
}
on = {
    "Toronto Region": ["City of Toronto Health Unit"],
    "Central Region": ["Peel Regional Health Unit", "York Regional Health Unit",
                       "Simcoe Muskoka District Health Unit", "Halton Regional Health Unit"],
    "East Region": ["Durham Regional Health Unit", "Haliburton, Kawartha, Pine Ridge District Health Unit",
                    "Peterborough County-City Health Unit", "Hastings and Prince Edward Counties Health Unit",
                    "Kingston, Frontenac and Lennox and Addington Health Unit",
                    "Leeds, Grenville and Lanark District Health Unit", "Renfrew County and District Health Unit",
                    "Eastern Ontario Health Unit", "City of Ottawa Health Unit"],
    "West Region": ["Brant County Health Unit", "Chatham-Kent Health Unit", "City of Hamilton Health Unit",
                    "Grey Bruce Health Unit", "Haldimand-Norfolk Health Unit", "Huron Perth Health Unit",
                    "Lambton Health Unit", "Middlesex-London Health Unit", "Niagara Regional Area Health Unit",
                    "Oxford Elgin St. Thomas Health Unit", "Waterloo Health Unit",
                    "Wellington-Dufferin-Guelph Health Unit", "Windsor-Essex County Health Unit"],
    "North East Region": ["District of Algoma Health Unit", "North Bay Parry Sound District Health Unit",
                          "Porcupine Health Unit", "Sudbury and District Health Unit", "Timiskaming Health Unit"],
    "North West Region": ["Northwestern Health Unit", "Thunder Bay District Health Unit"],
}

rows = combine(bc, "British Columbia", " Health Service Delivery Area") + combine(on, "Ontario")
rows.append({"province": "Prince Edward Island", "region": "Health PEI", **p.loc["Prince Edward Island"].to_dict()})

names = {"Arthritis": "arthritis_pct", "Perceived health, fair or poor": "fair_poor_health_pct",
         "Body mass index, adjusted self-reported, obese": "obese_pct"}
extra = pd.DataFrame(rows).rename(columns=names)
extra["survey_region"] = extra["region"]

cols = ["province", "survey_region", "arthritis_pct", "fair_poor_health_pct", "obese_pct"]
wide2 = pd.concat([wide[~wide["province"].isin(["British Columbia", "Ontario"])][cols], extra[cols]])
wide2["key"] = wide2["survey_region"].map(make_key)
regions["key"] = regions["region"].map(make_key)

final = regions.merge(wide2.drop(columns="survey_region"), on=["province", "key"], how="left", validate="one_to_one")
print(final.shape)
print(final[final["arthritis_pct"].isna()][["province", "region"]])
final[final["region"].str.contains("Edmonton")]

(49, 14)
Empty DataFrame
Columns: [province, region]
Index: []


,province,region,pct_meeting_benchmark,median_wait_days,p90_wait_days,volume,rank_shortest,region_key,family_docs_per_100k,specialists_per_100k,key,arthritis_pct,fair_poor_health_pct,obese_pct
2,Alberta,Edmonton Zone,79.2,95.0,257.0,903.0,7,edmonton zone,111.828469,156.203872,edmonton,21.4,33.1,15.0


In [12]:
final = final.drop(columns=["key", "region_key"])
final.to_csv("../data/processed/regions_2024_features.csv", index=False)

cols = ["median_wait_days", "family_docs_per_100k", "specialists_per_100k",
        "arthritis_pct", "fair_poor_health_pct", "obese_pct"]
final[cols].corr().round(2)

,median_wait_days,family_docs_per_100k,specialists_per_100k,arthritis_pct,fair_poor_health_pct,obese_pct
median_wait_days,1.00,-0.18,-0.16,0.07,0.19,-0.09
family_docs_per_100k,-0.18,1.00,0.44,-0.08,-0.21,-0.03
specialists_per_100k,-0.16,0.44,1.00,-0.31,-0.34,-0.10
arthritis_pct,0.07,-0.08,-0.31,1.00,0.42,0.33
fair_poor_health_pct,0.19,-0.21,-0.34,0.42,1.00,-0.53
obese_pct,-0.09,-0.03,-0.10,0.33,-0.53,1.00


## What I did and found

### What I did
Added three population-health measures from Statistics Canada's Health characteristics survey (2023/2024): arthritis, fair or poor perceived health, and obesity. Matching regions took work: names differed between sources, and BC's 16 small areas and Ontario's 34 health units were combined into our 5 BC health authorities and 6 Ontario regions, weighting by population. All 49 regions matched.

### What I found
- None of the three new predictors relates strongly to hip replacement waits (correlations between -0.09 and 0.19).
- Fair or poor health is the strongest (0.19): sicker regions wait slightly longer, but the link is weak.
- Predictors overlap moderately but not enough to remove any.

### Limits
The survey covers 2023/2024 while waits are 2024. Ontario's health units were assigned to the region where most residents live, so Ontario values are approximate. Survey values are self-reported.